# Week 4: Predictive Modeling and Optimization in Logistics Systems

**Target:** `Actual_Shipping_Days`

`Delay_Days` and `Late_Delivery_Flag` are excluded because they are derived from the target and would cause target leakage.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

df=pd.read_csv("../data/week2_cleaned_logistics_data.csv")
df.head()

In [ ]:
target="Actual_Shipping_Days"
features=["Shipment_Quantity","Product_Price","Shipping_Cost","Scheduled_Shipping_Days","Customer_Rating",
          "Shipping_Mode","Region","Customer_Segment","Product_Category"]
X=df[features]
y=df[target]
nums=["Shipment_Quantity","Product_Price","Shipping_Cost","Scheduled_Shipping_Days","Customer_Rating"]
cats=["Shipping_Mode","Region","Customer_Segment","Product_Category"]
prep=ColumnTransformer([
    ("num",SimpleImputer(strategy="median"),nums),
    ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                     ("oh",OneHotEncoder(handle_unknown="ignore"))]),cats)
])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)

In [ ]:
models={
"Linear Regression":LinearRegression(),
"Decision Tree":DecisionTreeRegressor(max_depth=5,min_samples_leaf=3,random_state=42),
"Random Forest":RandomForestRegressor(n_estimators=200,max_depth=8,min_samples_leaf=2,random_state=42)}
results=[]
for name,m in models.items():
    pipe=Pipeline([("prep",prep),("model",m)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    results.append({"Model":name,"MAE":mean_absolute_error(y_test,pred),
                    "RMSE":np.sqrt(mean_squared_error(y_test,pred)),"R2":r2_score(y_test,pred)})
pd.DataFrame(results).sort_values("RMSE")

In [ ]:
cv=KFold(n_splits=5,shuffle=True,random_state=42)
for name,m in models.items():
    pipe=Pipeline([("prep",prep),("model",m)])
    s=cross_val_score(pipe,X,y,cv=cv,scoring="neg_root_mean_squared_error")
    print(name,"CV RMSE:",-s.mean(),"+/-",s.std())

In [ ]:
rf=Pipeline([("prep",prep),("model",RandomForestRegressor(random_state=42))])
grid=GridSearchCV(rf,{
"model__n_estimators":[100,200],
"model__max_depth":[5,8,None],
"model__min_samples_leaf":[1,2,4]},cv=cv,scoring="neg_root_mean_squared_error",n_jobs=-1)
grid.fit(X_train,y_train)
pred=grid.predict(X_test)
print("Best parameters:",grid.best_params_)
print("MAE:",mean_absolute_error(y_test,pred))
print("RMSE:",np.sqrt(mean_squared_error(y_test,pred)))
print("R2:",r2_score(y_test,pred))

## Optimization Strategies
- Use predicted delivery time as an early-warning signal for potentially late shipments.
- Allocate warehouse, dispatch, and transport resources based on predicted workload.
- Compare predicted delivery duration and transportation cost when selecting shipping modes.
- Combine predictions with distance, traffic, and route history for route planning.
- Balance faster delivery against transportation cost for cost-aware decisions.
- Retrain and monitor the model periodically as new shipment records become available.